[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dandi/example-notebooks/blob/master/tutorials/cosyne_2023/simple_dandiset_search.ipynb)

## Installing requirements

The cell below installs every Python package needed to run this notebook, at fully pinned versions, using [`uv`](https://github.com/astral-sh/uv) for fast resolution. In Colab the cell is collapsed by default — click the ▶ button to run it.

In [ ]:
#@title Installing requirements (click ▶ to run) { display-mode: "form" }
# Colab provides Python 3.12. We install with `uv --system` because Colab's
# kernel runs outside a virtualenv. All versions (direct + transitive) are
# pinned below so the notebook is reproducible regardless of resolver drift.
!pip install -q uv
!uv pip install --system \
    "acres==0.5.0" \
    "aiohappyeyeballs==2.6.1" \
    "aiohttp==3.13.5" \
    "aiosignal==1.4.0" \
    "annotated-types==0.7.0" \
    "arrow==1.4.0" \
    "asciitree==0.3.3" \
    "attrs==26.1.0" \
    "backcall==0.2.0" \
    "bids-validator-deno==2.4.1" \
    "bidsschematools==1.2.2" \
    "blessed==1.38.0" \
    "certifi==2026.4.22" \
    "cffi==2.0.0" \
    "charset-normalizer==3.4.7" \
    "ci-info==0.4.0" \
    "click==8.1.8" \
    "click-didyoumean==0.3.1" \
    "comm==0.2.3" \
    "cryptography==43.0.3" \
    "dandi==0.75.1" \
    "dandischema==0.12.1" \
    "decorator==4.4.2" \
    "deprecated==1.3.1" \
    "dnspython==2.8.0" \
    "email-validator==2.3.0" \
    "etelemetry==0.3.1" \
    "fasteners==0.20" \
    "fqdn==1.5.1" \
    "frozenlist==1.8.0" \
    "fscacher==0.4.4" \
    "fsspec==2025.3.0" \
    "h5py==3.16.0" \
    "hdmf==4.3.1" \
    "hdmf-zarr==0.12.0" \
    "humanize==4.15.0" \
    "idna==3.13" \
    "interleave==0.3.0" \
    "ipython==7.34.0" \
    "ipywidgets==8.1.8" \
    "isodate==0.7.2" \
    "isoduration==20.11.0" \
    "jaraco-classes==3.4.0" \
    "jaraco-context==6.1.2" \
    "jaraco-functools==4.4.0" \
    "jedi==0.20.0" \
    "jeepney==0.9.0" \
    "joblib==1.5.3" \
    "jsonpointer==3.1.1" \
    "jsonschema==4.26.0" \
    "jsonschema-specifications==2025.9.1" \
    "jupyterlab-widgets==3.0.16" \
    "keyring==25.7.0" \
    "keyrings-alt==5.0.2" \
    "matplotlib-inline==0.2.1" \
    "ml-dtypes==0.5.4" \
    "more-itertools==10.8.0" \
    "multidict==6.7.1" \
    "natsort==8.4.0" \
    "numcodecs==0.15.1" \
    "numpy==2.0.2" \
    "nwbinspector==0.7.1" \
    "packaging==26.1" \
    "pandas==2.2.2" \
    "parso==0.8.6" \
    "pexpect==4.9.0" \
    "pickleshare==0.7.5" \
    "platformdirs==4.9.6" \
    "prompt-toolkit==3.0.52" \
    "propcache==0.4.1" \
    "ptyprocess==0.7.0" \
    "pycparser==3.0" \
    "pycryptodomex==3.23.0" \
    "pydantic==2.12.3" \
    "pydantic-core==2.41.4" \
    "pydantic-settings==2.14.0" \
    "pygments==2.20.0" \
    "pynwb==3.1.3" \
    "pyout==0.8.1" \
    "python-dateutil==2.9.0.post0" \
    "python-dotenv==1.2.2" \
    "pytz==2025.2" \
    "pyyaml==6.0.3" \
    "referencing==0.37.0" \
    "requests==2.32.4" \
    "rfc3339-validator==0.1.4" \
    "rfc3987==1.3.8" \
    "rpds-py==0.30.0" \
    "ruamel-yaml==0.19.1" \
    "secretstorage==3.5.0" \
    "semantic-version==2.10.0" \
    "setuptools==75.2.0" \
    "six==1.17.0" \
    "tenacity==9.1.4" \
    "tensorstore==0.1.82" \
    "threadpoolctl==3.6.0" \
    "tqdm==4.67.3" \
    "traitlets==5.7.1" \
    "typing-extensions==4.15.0" \
    "typing-inspection==0.4.2" \
    "tzdata==2026.1" \
    "uri-template==1.3.0" \
    "urllib3==2.5.0" \
    "wcwidth==0.6.0" \
    "webcolors==25.10.0" \
    "widgetsnbextension==4.0.15" \
    "wrapt==2.1.2" \
    "yarl==1.23.0" \
    "zarr==2.18.7" \
    "zarr-checksum==0.4.7"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already loaded in the kernel. Go to **Runtime → Restart session**, then **Run all cells below** (skip this install cell on re-run).

In [1]:
import json
from dandi.dandiapi import DandiAPIClient
from tqdm.notebook import tqdm

In [2]:
client = DandiAPIClient()
dandisets = list(client.get_dandisets())

# Identify NWB dandisets
Most dandisets hold NWB-formatted data, but DANDI also hold data of other formats.

Let's start by filtering down to only the dandisets that contain at least one NWB file.

We can do this by querying the metadata of each dandiset, which tells us the data formats within in `raw_metadata["assetsSummary"]["dataStandard"]`.

If no data has been uploaded to that dandiset, the "dataStandard" field is not present.

We handle this by using the `.get` method to iterate over an empty list.

In [3]:
nwb_dandisets = []

for dandiset in tqdm(dandisets):
    raw_metadata = dandiset.get_raw_metadata()

    if any(
        "NWB" in data_standard["name"]
        for data_standard in raw_metadata["assetsSummary"].get("dataStandard", [])
    ):
        nwb_dandisets.append(dandiset)
print(f"There are currently {len(nwb_dandisets)} NWB datasets on DANDI!")

  0%|          | 0/223 [00:00<?, ?it/s]

There are currently 128 NWB datasets on DANDI!


# Filtering dandisets: species
Let's use the `nwb_dandisets` list from the previous recipe and see which of them used mice in their study.

You can find this information in `raw_metadata["assetsSummary"]["species"]`.

We'll use the same `.get` trick as above for if no data has been uploaded.

In [4]:
mouse_nwb_dandisets = []

for dandiset in tqdm(nwb_dandisets):
    raw_metadata = dandiset.get_raw_metadata()

    if any(
        "mouse" in species["name"]
        for species in raw_metadata["assetsSummary"].get("species", [])
    ):
        mouse_nwb_dandisets.append(dandiset)
print(f"There are currently {len(mouse_nwb_dandisets)} NWB datasets on DANDI that use mice!")

  0%|          | 0/128 [00:00<?, ?it/s]

There are currently 61 NWB datasets on DANDI that use mice!


# Filtering by session: species and sex
Let's say you have identified a dandiset of interest, "000005", and you want to identify all of the sessions on female mice.

You can do this by querying asset-level metadata.

Assets correspond to individual NWB files, and contain metadata extracted from those files.

The metadata of each asset contains a `.wasAttributedTo` attribute, which is a list of `Participant` objects corresponding to the subjects for that session.

We do that by first testing that attribute exists (is not `None` - some older dandisets may not have included it) and then checking the value of its `name` parameter.

In [5]:
dandiset = client.get_dandiset("000005")
female_mouse_nwb_sessions = []

assets = list(dandiset.get_assets())
for asset in tqdm(assets):
    asset_metadata = asset.get_metadata()
    subjects = asset_metadata.wasAttributedTo

    if any(
        subject.species and "mouse" in subject.species.name.lower()
        and subject.sex and subject.sex.name == "Female"
        for subject in subjects
    ):
        female_mouse_nwb_sessions.append(asset)
print(f"Dandiset #5 has {len(female_mouse_nwb_sessions)} out of {len(assets)} files that use female mice!")

  0%|          | 0/148 [00:00<?, ?it/s]

Dandiset #5 has 69 out of 148 files that use female mice!


# Going beyond
These examples show a few types of queries, but since the metadata structures are quite rich on both the dandiset and asset levels, they enable many complex queries beyond the examples here.

These metadata structures are also expanding over time as DANDI becomes more strict about what counts as essential metadata.

The `.get_raw_metadata` method of both `client.get_dandiset(...)` and `client.get_dandiset(...).get_assets()` provides a nice view into the available fields.

Note: for any attribute, it is recommended to first check that it is not `None` before checking for its value.

In [6]:
print(json.dumps(dandisets[0].get_raw_metadata(), indent=4))

{
    "id": "DANDI:000003/0.210812.1448",
    "doi": "10.48324/dandi.000003/0.210812.1448",
    "url": "https://dandiarchive.org/dandiset/000003/0.210812.1448",
    "name": "Physiological Properties and Behavioral Correlates of Hippocampal Granule Cells and Mossy Cells",
    "about": [
        {
            "name": "hippocampus",
            "schemaKey": "Anatomy",
            "identifier": "UBERON:0002421"
        }
    ],
    "access": [
        {
            "status": "dandi:OpenAccess",
            "schemaKey": "AccessRequirements",
            "contactPoint": {
                "email": "petersen.peter@gmail.com",
                "schemaKey": "ContactPoint"
            }
        }
    ],
    "license": [
        "spdx:CC-BY-4.0"
    ],
    "version": "0.210812.1448",
    "@context": "https://raw.githubusercontent.com/dandi/schema/master/releases/0.4.4/context.json",
    "citation": "Senzai, Yuta; Fernandez-Ruiz, Antonio; Buzs\u00e1ki, Gy\u00f6rgy (2021) Physiological Properties and